


This notebook calculates real road network distance from each county centroid to its nearest pharmacy using OSMnx + Dijkstra's algorithm.




In [ ]:
import subprocess
packages = ['osmnx', 'libpysal', 'esda', 'tqdm']
for pkg in packages:
    result = subprocess.run(['pip', 'install', pkg, '-q'], capture_output=True, text=True)
    print(f'{pkg}: {"OK" if result.returncode == 0 else result.stderr}')

#Imports and Configuration

In [ ]:
import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
from math import radians
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── PATHS ───────────────────────────────────────────────
BASE_DIR      = os.path.expanduser('~/P')
PHARMACY_FILE = f'{BASE_DIR}/Pharmacy (3).csv'
MASTER_FILE   = f'{BASE_DIR}/master_with_lisa.csv'
SHP_FILE      = f'{BASE_DIR}/county-2020-500k.shp'
OUTPUT_FILE   = f'{BASE_DIR}/osmnx_results.csv'
CHECKPOINT    = f'{BASE_DIR}/osmnx_checkpoint.csv'
CACHE_DIR     = f'{BASE_DIR}/osmnx_cache'

# ── OSMnx SETTINGS ──────────────────────────────────────
ox.settings.log_console  = False
ox.settings.use_cache    = True
ox.settings.cache_folder = CACHE_DIR
os.makedirs(CACHE_DIR, exist_ok=True)

# ── PARAMETERS ──────────────────────────────────────────
TOP_N      = 5    # haversine candidates per county
SLEEP_SEC  = 0.3  # pause between counties
CHECKPOINT_EVERY = 25  # save every N counties

print('All imports successful')
print(f'Base directory : {BASE_DIR}')
ram = psutil.virtual_memory()
print(f'Available RAM  : {ram.available / (1024**3):.1f} GB')

 Load Data and Build County Centroids

In [ ]:
print('Loading pharmacies...')
pharmacies = pd.read_csv(PHARMACY_FILE)
print(f'  Pharmacies loaded : {len(pharmacies)}')

print('Loading master table...')
master = pd.read_csv(MASTER_FILE, dtype={'county_fips': str, 'state_fips': str})
print(f'  Master rows       : {len(master)}')

print('Loading shapefile...')
os.environ['SHAPE_RESTORE_SHX'] = 'YES'
shp = gpd.read_file(SHP_FILE)
shp = shp.rename(columns={'GEOID': 'county_fips'})
print(f'  Shapefile rows    : {len(shp)}')

# Merge shapefile with master
merged = shp.merge(master, on='county_fips', how='inner')

# Only counties with real death rate data
merged = merged.dropna(subset=['avg_crude_rate'])

# Extract centroids
merged = merged.copy()
merged['cent_lat'] = merged.geometry.centroid.y
merged['cent_lon'] = merged.geometry.centroid.x

# Drop Alaska, Hawaii, Puerto Rico for cleaner analysis
merged = merged[~merged['state_fips'].isin(['02', '15', '72'])]

print(f'\nCounties to process : {len(merged)}')
print(merged[['county_fips', 'cent_lat', 'cent_lon']].head())

Haversine Function

In [ ]:
# Pre-extract pharmacy arrays for fast vectorized computation
pharm_lats = pharmacies['lat'].values
pharm_lons = pharmacies['lon'].values

def get_top_n_candidates(cent_lat, cent_lon, n=TOP_N):
    """
    Returns indices and distances of n nearest pharmacies
    using vectorized haversine (straight line distance).
    Used to narrow down candidates before OSMnx.
    """
    R    = 6371
    la1  = radians(cent_lat)
    lo1  = radians(cent_lon)
    la2  = np.radians(pharm_lats)
    lo2  = np.radians(pharm_lons)
    dlat = la2 - la1
    dlon = lo2 - lo1
    a    = np.sin(dlat/2)**2 + np.cos(la1) * np.cos(la2) * np.sin(dlon/2)**2
    dist = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    idx  = np.argpartition(dist, n)[:n]
    return idx, dist[idx]

# Quick test
test_idx, test_dists = get_top_n_candidates(37.4, -81.6)
print('Haversine function working')
print(f'Nearest pharmacy to (37.4, -81.6): {pharmacies.iloc[test_idx[np.argmin(test_dists)]]["name"]} at {test_dists.min():.1f} km')

Check / Resume From Checkpoint

In [ ]:
if os.path.exists(CHECKPOINT):
    done_df   = pd.read_csv(CHECKPOINT, dtype={'county_fips': str})
    completed = set(done_df['county_fips'].values)
    results   = done_df.to_dict('records')
    print(f'Resuming from checkpoint')
    print(f'  Already done : {len(completed)} counties')
else:
    completed = set()
    results   = []
    print('Starting fresh — no checkpoint found')

remaining = merged[~merged['county_fips'].isin(completed)]
print(f'  Remaining    : {len(remaining)} counties')
print(f'  Est. time    : {len(remaining) * 12 / 3600:.1f} hours')

Main OSMnx Loop

In [ ]:
print(f'Starting OSMnx loop for {len(remaining)} counties...')
print(f'Checkpointing every {CHECKPOINT_EVERY} counties to {CHECKPOINT}')
print('='*60)

for _, row in tqdm(remaining.iterrows(), total=len(remaining), desc='Counties'):

    fips     = row['county_fips']
    cent_lat = row['cent_lat']
    cent_lon = row['cent_lon']
    G        = None

    try:
        # ── STEP 1: Haversine candidates ──────────────────
        idx, dists       = get_top_n_candidates(cent_lat, cent_lon)
        top_ph           = pharmacies.iloc[idx].copy()
        top_ph['str_km'] = dists
        nearest_straight = dists.min()

        # ── STEP 2: Dynamic radius ─────────────────────────
        # Straight line × 2.5 safety buffer, minimum 10km
        radius = int(max(nearest_straight * 2.5 * 1000, 10000))

        # ── STEP 3: Download road network ──────────────────
        G = ox.graph_from_point(
            (cent_lat, cent_lon),
            dist=radius,
            network_type='drive'
        )

        # ── STEP 4: Snap centroid to road network ──────────
        centroid_node = ox.nearest_nodes(G, cent_lon, cent_lat)

        # ── STEP 5: Dijkstra to each candidate pharmacy ────
        best_road_km  = np.inf
        best_name     = None
        best_straight = None

        for _, p in top_ph.iterrows():
            try:
                p_node = ox.nearest_nodes(G, p['lon'], p['lat'])
                path_m = nx.shortest_path_length(
                    G, centroid_node, p_node, weight='length'
                )
                road_km = path_m / 1000
                if road_km < best_road_km:
                    best_road_km  = road_km
                    best_name     = p['name']
                    best_straight = p['str_km']
            except:
                continue

        # ── STEP 6: Store result ───────────────────────────
        results.append({
            'county_fips'        : fips,
            'nearest_pharm_name' : best_name,
            'straight_line_km'   : round(float(best_straight), 3) if best_straight is not None else None,
            'road_distance_km'   : round(best_road_km, 3) if best_road_km != np.inf else None,
            'method'             : 'osmnx'
        })

    except Exception as e:
        # ── FALLBACK: OSMnx failed, use haversine ──────────
        try:
            idx, dists = get_top_n_candidates(cent_lat, cent_lon)
            best_idx   = np.argmin(dists)
            results.append({
                'county_fips'        : fips,
                'nearest_pharm_name' : pharmacies.iloc[idx[best_idx]]['name'],
                'straight_line_km'   : round(float(dists[best_idx]), 3),
                'road_distance_km'   : None,
                'method'             : 'haversine_fallback'
            })
        except:
            results.append({
                'county_fips'        : fips,
                'nearest_pharm_name' : None,
                'straight_line_km'   : None,
                'road_distance_km'   : None,
                'method'             : 'failed'
            })

    finally:
        # ── MEMORY CLEANUP ─────────────────────────────────
        if G is not None:
            del G
        gc.collect()
        time.sleep(SLEEP_SEC)

    # ── CHECKPOINT SAVE ────────────────────────────────────
    if len(results) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)

print('\nLoop complete!')

Save Final Results

In [ ]:
final_df = pd.DataFrame(results)

# Save to ~/P/
final_df.to_csv(OUTPUT_FILE, index=False)

print('='*60)
print(f'RESULTS SAVED TO: {OUTPUT_FILE}')
print('='*60)
print(f'Total counties processed : {len(final_df)}')
print(f'OSMnx success            : {(final_df["method"] == "osmnx").sum()}')
print(f'Haversine fallback       : {(final_df["method"] == "haversine_fallback").sum()}')
print(f'Failed                   : {(final_df["method"] == "failed").sum()}')
print()
print('Sample results:')
print(final_df.head(10).to_string())
print()
print('Distance stats (road_distance_km):')
print(final_df['road_distance_km'].describe())